In [1]:
import shutil
shutil.rmtree("/kaggle/working/semantickitti_subsampled", ignore_errors=True)
print("Cleared!")

# Check free space
import subprocess
result = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print(result.stdout)

Cleared!
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  128K   20G   1% /kaggle/working



In [2]:
'''
import numpy as np
from pathlib import Path

SRC = Path("/kaggle/input/datasets/parthibhang/semantickitti/sequences")
RARE_CLASSES = {30,31,32,252,253,254,255,256,257,258,259}

def get_rare_frames(label_dir):
    rare = set()
    for i, lf in enumerate(sorted(Path(label_dir).glob("*.label"))):
        labels = np.fromfile(lf, dtype=np.uint32) & 0xFFFF
        if RARE_CLASSES & set(labels.tolist()):
            rare.add(i)
    return rare

# Try different stride combinations
for train_stride, rare_stride in [(5,5), (6,6), (6,8), (7,7)]:
    total_kept = 0
    total_frames = 0
    
    SPLITS = {
        "train": {"seqs": ["00","01","02","03","04","05","06","07","09","10"], 
                  "stride": train_stride, "rare_stride": rare_stride, "has_labels": True},
        "valid": {"seqs": ["08"], 
                  "stride": 2, "rare_stride": 1, "has_labels": True},
        "test":  {"seqs": ["11","12","13","14","15","16","17","18","19","20","21"], 
                  "stride": 8, "rare_stride": 8, "has_labels": False},
    }
    
    for split_name, cfg in SPLITS.items():
        for seq in cfg["seqs"]:
            src_seq = SRC / seq
            if not src_seq.exists(): continue
            stems = [f.stem for f in sorted((src_seq/"velodyne").glob("*.bin"))]
            
            rare_set = set()
            if cfg["has_labels"] and (src_seq/"labels").exists():
                all_rare = sorted(get_rare_frames(src_seq/"labels"))
                rare_set = set(all_rare[i] for i in range(0, len(all_rare), cfg["rare_stride"]))
            
            kept = sum(1 for i in range(len(stems))
                       if (i % cfg["stride"] == 0) or (i in rare_set))
            total_kept += kept
            total_frames += len(stems)
    
    est_gb = total_kept * 1.5 / 1000
    pct = 100 * total_kept / total_frames
    print(f"train_stride={train_stride}, rare_stride={rare_stride} → {total_kept} frames ({pct:.1f}%) ~{est_gb:.1f}GB")
    '''

'\nimport numpy as np\nfrom pathlib import Path\n\nSRC = Path("/kaggle/input/datasets/parthibhang/semantickitti/sequences")\nRARE_CLASSES = {30,31,32,252,253,254,255,256,257,258,259}\n\ndef get_rare_frames(label_dir):\n    rare = set()\n    for i, lf in enumerate(sorted(Path(label_dir).glob("*.label"))):\n        labels = np.fromfile(lf, dtype=np.uint32) & 0xFFFF\n        if RARE_CLASSES & set(labels.tolist()):\n            rare.add(i)\n    return rare\n\n# Try different stride combinations\nfor train_stride, rare_stride in [(5,5), (6,6), (6,8), (7,7)]:\n    total_kept = 0\n    total_frames = 0\n    \n    SPLITS = {\n        "train": {"seqs": ["00","01","02","03","04","05","06","07","09","10"], \n                  "stride": train_stride, "rare_stride": rare_stride, "has_labels": True},\n        "valid": {"seqs": ["08"], \n                  "stride": 2, "rare_stride": 1, "has_labels": True},\n        "test":  {"seqs": ["11","12","13","14","15","16","17","18","19","20","21"], \n         

In [3]:
import os, shutil, numpy as np
from pathlib import Path

SRC = Path("/kaggle/input/datasets/parthibhang/semantickitti/sequences")
DST = Path("/kaggle/working/semantickitti_subsampled/sequences")

RARE_CLASSES = {30,31,32,252,253,254,255,256,257,258,259}

SPLITS = {
    "train": {"seqs": ["00","01","02","03","04","05","06","07","09","10"], 
              "stride": 10, "rare_stride": 8, "has_labels": True},
    "valid": {"seqs": ["08"], 
              "stride": 10, "rare_stride": 6, "has_labels": True},
    "test":  {"seqs": ["11","12","13","14","15","16","17","18","19","20","21"], 
              "stride": 12, "rare_stride": 10, "has_labels": False},
}

def get_rare_frames(label_dir):
    rare = set()
    for i, lf in enumerate(sorted(Path(label_dir).glob("*.label"))):
        labels = np.fromfile(lf, dtype=np.uint32) & 0xFFFF
        if RARE_CLASSES & set(labels.tolist()):
            rare.add(i)
    return rare

def copy_frame(src_seq, dst_seq, stem, has_labels):
    src_bin = src_seq / "velodyne" / f"{stem}.bin"
    dst_bin = dst_seq / "velodyne" / f"{stem}.bin"
    if src_bin.exists():
        shutil.copy2(src_bin, dst_bin)
    if has_labels:
        src_lbl = src_seq / "labels" / f"{stem}.label"
        dst_lbl = dst_seq / "labels" / f"{stem}.label"
        if src_lbl.exists():
            shutil.copy2(src_lbl, dst_lbl)

def copy_metadata(src_seq, dst_seq):
    for item in src_seq.iterdir():
        if item.is_file():
            shutil.copy2(item, dst_seq / item.name)

for split_name, cfg in SPLITS.items():
    print(f"\n=== {split_name.upper()} ===")
    for seq in cfg["seqs"]:
        src_seq = SRC / seq
        dst_seq = DST / seq
        if not src_seq.exists():
            print(f"  [SKIP] {seq}"); continue

        (dst_seq / "velodyne").mkdir(parents=True, exist_ok=True)
        if cfg["has_labels"]:
            (dst_seq / "labels").mkdir(parents=True, exist_ok=True)
        copy_metadata(src_seq, dst_seq)

        stems = [f.stem for f in sorted((src_seq/"velodyne").glob("*.bin"))]

        rare_set = set()
        if cfg["has_labels"] and (src_seq/"labels").exists():
            all_rare = sorted(get_rare_frames(src_seq/"labels"))
            rare_set = set(all_rare[i] for i in range(0, len(all_rare), cfg["rare_stride"]))

        kept = 0
        for i, stem in enumerate(stems):
            if (i % cfg["stride"] == 0) or (i in rare_set):
                copy_frame(src_seq, dst_seq, stem, cfg["has_labels"])
                kept += 1

        pct = 100 * kept / len(stems) if stems else 0
        print(f"  seq {seq}: {kept}/{len(stems)} ({pct:.1f}%)")

# Final size check
import subprocess
result = subprocess.run(["du", "-sh", "/kaggle/working/semantickitti_subsampled"],
                        capture_output=True, text=True)
print(f"\nFinal size: {result.stdout.strip()}")
print("\nDone!")


=== TRAIN ===
  seq 00: 665/4541 (14.6%)
  seq 01: 206/1101 (18.7%)
  seq 02: 588/4661 (12.6%)
  seq 03: 114/801 (14.2%)
  seq 04: 55/271 (20.3%)
  seq 05: 454/2761 (16.4%)
  seq 06: 178/1101 (16.2%)
  seq 07: 200/1101 (18.2%)
  seq 09: 267/1591 (16.8%)
  seq 10: 186/1201 (15.5%)

=== VALID ===
  seq 08: 871/4071 (21.4%)

=== TEST ===
  seq 11: 77/921 (8.4%)
  seq 12: 89/1061 (8.4%)
  seq 13: 274/3281 (8.4%)
  seq 14: 53/631 (8.4%)
  seq 15: 159/1901 (8.4%)
  seq 16: 145/1731 (8.4%)
  seq 17: 41/491 (8.4%)
  seq 18: 151/1801 (8.4%)
  seq 19: 416/4981 (8.4%)
  seq 20: 70/831 (8.4%)
  seq 21: 227/2721 (8.3%)

Final size: 12G	/kaggle/working/semantickitti_subsampled

Done!


In [4]:
!zip -r semkitti_research.zip /kaggle/working/semantickitti_subsampled

  adding: kaggle/working/semantickitti_subsampled/ (stored 0%)
  adding: kaggle/working/semantickitti_subsampled/sequences/ (stored 0%)
  adding: kaggle/working/semantickitti_subsampled/sequences/21/ (stored 0%)
  adding: kaggle/working/semantickitti_subsampled/sequences/21/velodyne/ (stored 0%)
  adding: kaggle/working/semantickitti_subsampled/sequences/21/velodyne/001620.bin (deflated 26%)
  adding: kaggle/working/semantickitti_subsampled/sequences/21/velodyne/000288.bin (deflated 26%)
  adding: kaggle/working/semantickitti_subsampled/sequences/21/velodyne/000048.bin (deflated 25%)
  adding: kaggle/working/semantickitti_subsampled/sequences/21/velodyne/001752.bin (deflated 26%)
  adding: kaggle/working/semantickitti_subsampled/sequences/21/velodyne/001548.bin (deflated 26%)
  adding: kaggle/working/semantickitti_subsampled/sequences/21/velodyne/001140.bin (deflated 26%)
  adding: kaggle/working/semantickitti_subsampled/sequences/21/velodyne/000264.bin (deflated 26%)
  adding: kaggle/